# Session 4: fit a curved Arrhenius plot
**A. Real-world problem.** Session 3 ended with a warning: when two reactions run in parallel,
the plot of ln k against 1000/T bends, and a straight line predicts k at 2000 K about nine to
fifty times too low (it depends how cold the data start) while its own band says it is
sure. A combustion or pyrolysis model needs the right number, so we fit **two channels**
and report how sure we are.

**Plan.** (1) the two-channel model, (2) fit it by nonlinear least squares (Levenberg-Marquardt,
written out in numpy), (3) the covariance and why the parameters are hard to pin down while the
prediction is not, (4) the band at 2000 K, (5) is the extra channel justified? (6) validate
against known truth, (7) three ways it fails: data that stop too cool, bad start values,
too few points.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

R_GAS = 8.314462618          # J/(mol K)
# the same two-channel truth as Session 3: lnA1, E1 (kJ/mol), lnA2, E2 (kJ/mol)
A1, E1 = 1.0e7, 50.0         # low-Ea channel, wins when it is cool
A2, E2 = 1.0e14, 200.0       # high-Ea channel, wins when it is hot
P_TRUE = np.array([np.log(A1), E1, np.log(A2), E2])
T_DATA = np.linspace(500.0, 1600.0, 12)     # K, the roadmap data window
T_TARGET = 2000.0                            # K, where the engine model needs k

# two-sided 95 percent Student t factors by degrees of freedom
_T95 = {1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776, 5: 2.571, 6: 2.447, 7: 2.365, 8: 2.306, 9: 2.262,
        10: 2.228, 12: 2.179, 15: 2.131, 20: 2.086, 30: 2.042}


def t95(df):
    """Student t factor for a 95 percent two-sided interval (table; next lower tabulated df, so slightly wide)."""
    if df >= 30:
        return 2.042 if df < 120 else 1.98
    return _T95[max(d for d in _T95 if d <= df)]

## Step 1: the model
Each channel is an Arrhenius term, so with x = 1000/T and E in kJ/mol

    k(T) = exp(lnA1 - E1 x / R) + exp(lnA2 - E2 x / R)

(R = 8.314 J/(mol K), E in kJ/mol, x = 1000/T: the 1000 that turns kJ into J cancels the 1000 in
x, so the exponent is simply E x / R). We fit **ln k**, as in Sessions 1 to 3,
so a 3 percent error is a 0.03 error in ln k at every temperature. ln(a + b) = logaddexp(ln a, ln b),
which never overflows.

The cool end of the plot is the low-Ea channel (a shallow line), the hot end is the high-Ea
channel (a steep line), and the **crossover** is where the two terms are equal.

In [ ]:
def two_channel_k(T, p=P_TRUE):
    """k (1/s) from p = (lnA1, E1, lnA2, E2)."""
    return np.exp(ln_k(p, T))


def ln_k(p, T):
    x = 1000.0 / np.asarray(T, dtype=float)
    return np.logaddexp(p[0] - p[1] * x / R_GAS, p[2] - p[3] * x / R_GAS)


def jacobian(p, T):
    """d ln k / d (lnA1, E1, lnA2, E2), shape (n, 4). The weights w1, w2 are each channel's share of k."""
    x = 1000.0 / np.asarray(T, dtype=float)
    t1, t2 = p[0] - p[1] * x / R_GAS, p[2] - p[3] * x / R_GAS
    y = np.logaddexp(t1, t2)
    w1, w2 = np.exp(t1 - y), np.exp(t2 - y)
    return np.column_stack([w1, -w1 * x / R_GAS, w2, -w2 * x / R_GAS])


def crossover_T(p):
    """Temperature (K) at which the two channels contribute equally; None if the lines never cross."""
    dE = p[1] - p[3]
    if dE == 0.0:
        return None
    x = R_GAS * (p[0] - p[2]) / dE
    return float(1000.0 / x) if x > 0 else None


def channel_share(p, T):
    """Fraction of k carried by the second (high-Ea) channel."""
    x = 1000.0 / np.asarray(T, dtype=float)
    return np.exp(p[2] - p[3] * x / R_GAS - ln_k(p, T))


def synthetic(T, rel_noise=0.03, seed=0, p=P_TRUE):
    rng = np.random.default_rng(seed)
    T = np.asarray(T, dtype=float)
    return two_channel_k(T, p) * np.exp(rel_noise * rng.standard_normal(T.size))

## Step 2: fitting it
Unlike a straight line there is no formula. Start from a guess p, linearise ln k around it with
the Jacobian J (the table of slopes d ln k / d p), solve for the best step, and repeat:

    (J^T J + lambda diag(J^T J)) step = J^T r,     r = ln k_data - ln k_model.

This is **Levenberg-Marquardt**: lambda large = a short careful step downhill, lambda small =
the full Gauss-Newton step. A step that lowers the residual sum of squares (RSS) is kept and
lambda shrinks; a step that does not is rejected and lambda grows. Columns are scaled to unit
length first, because ln A (about 30) and E (about 200) differ by a factor ten.

A start guess matters because the surface has flat valleys. We try three: split the data
into a cool and a hot part, fit a straight line to each, and use the lines as the two
channels. Then we keep whichever start ends with the smallest RSS.

In [ ]:
def _lm(y, T, p0, iters=400):
    p = np.array(p0, dtype=float)
    res = y - ln_k(p, T)
    cur = float(res @ res)
    lam = 1e-2
    for _ in range(iters):
        J = jacobian(p, T)
        s = np.sqrt((J ** 2).sum(axis=0)) + 1e-12
        Js = J / s
        A, g = Js.T @ Js, Js.T @ res
        moved = False
        for _try in range(60):
            try:
                step = np.linalg.solve(A + lam * np.diag(np.diag(A) + 1e-12), g) / s
            except np.linalg.LinAlgError:
                lam *= 10.0
                continue
            pn = p + step
            rn = y - ln_k(pn, T)
            new = float(rn @ rn)
            if np.isfinite(new) and new < cur:
                done = cur - new < 1e-14 * max(cur, 1e-300) or np.abs(step / s).max() < 1e-10
                p, res, cur, lam, moved = pn, rn, new, max(lam / 3.0, 1e-12), True
                break
            lam *= 4.0
        if not moved or done:
            break
    return p, cur


def start_values(T, k):
    """Three start guesses: split the data after 1/3, 1/2 and 2/3 of the points; a straight line on
    each part (ln k against 1000/T) gives one channel."""
    T, k = np.asarray(T, dtype=float), np.asarray(k, dtype=float)
    order = np.argsort(T)
    T, k = T[order], k[order]
    x, y = 1000.0 / T, np.log(k)                 # T rises, so x falls: the first points are the hot ones
    n = T.size
    out = []
    for h in sorted({max(2, n // 3), max(2, n // 2), max(2, (2 * n) // 3)}):
        h = min(h, n - 2)
        b1h, b0h = np.polyfit(x[:h], y[:h], 1)           # hot part
        b1c, b0c = np.polyfit(x[h:], y[h:], 1)           # cool part
        out.append([b0c, -b1c * R_GAS, b0h, -b1h * R_GAS])
    return out


def fit_two_channel(T, k, starts=None):
    """Nonlinear least squares of ln k. Needs at least 5 points (4 parameters + 1 degree of freedom).
    Returns dict: p (lnA1, E1, lnA2, E2 with E1 <= E2), cov (4x4), df, s (error of ln k), rss, n_starts."""
    T, k = np.asarray(T, dtype=float), np.asarray(k, dtype=float)
    n = T.size
    if n < 5:
        raise ValueError("need at least 5 points: 4 parameters and one degree of freedom to estimate the error")
    y = np.log(k)
    best = None
    starts = start_values(T, k) if starts is None else [list(s) for s in starts]
    for p0 in starts:
        p, rss = _lm(y, T, p0)
        if best is None or rss < best[1]:
            best = (p, rss)
    p, rss = best
    if p[1] > p[3]:                                       # label the cool channel 1: swap the channels
        p = p[[2, 3, 0, 1]]
    df = n - 4
    s = float(np.sqrt(rss / df))
    J = jacobian(p, T)
    sc = np.sqrt((J ** 2).sum(axis=0)) + 1e-300
    Js = J / sc
    with np.errstate(all="ignore"):
        cov = s ** 2 * np.linalg.pinv(Js.T @ Js, rcond=1e-14) / np.outer(sc, sc)
    return {"p": p, "cov": cov, "df": df, "s": s, "rss": float(rss), "n_starts": len(starts)}


def param_table(fit):
    """Standard errors and the correlation matrix of (lnA1, E1, lnA2, E2)."""
    sd = np.sqrt(np.maximum(np.diag(fit["cov"]), 0.0))
    with np.errstate(divide="ignore", invalid="ignore"):
        corr = fit["cov"] / np.outer(sd, sd)
    return {"se": sd, "corr": corr}

## Step 3: why the parameters are hard to identify
The covariance of the fitted parameters is s^2 (J^T J)^-1, the same idea as Session 3 with the
Jacobian in place of the two columns (1, x). Look at its correlation matrix: lnA1 and E1
move together (about +0.99), and so do lnA2 and E2. Raising E1 by a little and lnA1 by the
matching amount leaves ln k almost unchanged over the data; the fit can tell the *combination*
but not each one. The eigenvalues of J^T J (after scaling the columns) say the same thing:
the ratio of the largest to the smallest is the *condition number*, and a large one means a
long flat valley in which very different parameters fit almost equally well.

The prediction is a different matter. It is one particular combination of the four parameters,
and the data can fix that combination well even when each parameter alone is poorly known.

In [ ]:
def condition_number(fit, T):
    """Largest over smallest eigenvalue of the column-scaled J^T J (a large number = a flat valley)."""
    J = jacobian(fit["p"], T)
    Js = J / (np.sqrt((J ** 2).sum(axis=0)) + 1e-300)
    w = np.linalg.eigvalsh(Js.T @ Js)
    return float(w[-1] / max(w[0], 1e-300))

## Step 4: the band at any temperature
The delta method: at temperature T the fitted ln k has gradient g = d ln k / d p (one row of
the Jacobian), so its variance is g^T cov g, exactly like Session 3 but with four parameters.
The 95 percent interval is ln k +- t * se with t for n - 4 degrees of freedom, and on the k
axis it is a factor exp(+-t se). A Monte Carlo draw from the fitted covariance gives the same
answer when the band is narrow, and shows when it is not.

In [ ]:
def se_ln_k(fit, T):
    g = jacobian(fit["p"], np.atleast_1d(np.asarray(T, dtype=float)))
    return np.sqrt(np.maximum(np.einsum("ij,jk,ik->i", g, fit["cov"], g), 0.0))


def band(fit, T):
    """(central k, lower k, upper k): 95 percent interval for the mean ln k."""
    T = np.atleast_1d(np.asarray(T, dtype=float))
    mid = ln_k(fit["p"], T)
    h = t95(fit["df"]) * se_ln_k(fit, T)
    with np.errstate(over="ignore"):
        return np.exp(mid), np.exp(mid - h), np.exp(mid + h)


def mc_predict(fit, T_target, draws=40000, seed=7):
    """(median k, 2.5 percent, 97.5 percent) at T_target from parameter draws with the fitted covariance."""
    rng = np.random.default_rng(seed)
    p = rng.multivariate_normal(fit["p"], fit["cov"], size=draws, check_valid="ignore")
    k = np.exp(ln_k(p.T, T_target))
    lo, med, hi = np.percentile(k, [2.5, 50, 97.5])
    return float(med), float(lo), float(hi)

## Step 5: is the second channel justified?
More parameters always fit better, so we need a fair comparison. The **corrected Akaike
criterion** charges a price for each parameter:

    AICc = n ln(RSS / n) + 2 m + 2 m (m + 1) / (n - m - 1),    m = number of parameters.

A difference of more than about 10 in favour of one model is overwhelming. Also look at the
residuals of the straight line: a curve leaves them in a run of positive, then negative,
then positive again; noise would not.

In [ ]:
def line_fit(T, k):
    """Straight line ln k = b0 + b1 * 1000/T. Returns dict b, rss, s, df."""
    x, y = 1000.0 / np.asarray(T, dtype=float), np.log(np.asarray(k, dtype=float))
    b1, b0 = np.polyfit(x, y, 1)
    res = y - (b0 + b1 * x)
    rss = float(res @ res)
    return {"b": np.array([b0, b1]), "rss": rss, "df": x.size - 2, "s": float(np.sqrt(rss / (x.size - 2))), "res": res}


def line_k(fit, T):
    return np.exp(fit["b"][0] + fit["b"][1] * 1000.0 / np.asarray(T, dtype=float))


def aicc(rss, n, m):
    if n - m - 1 <= 0:
        raise ValueError("AICc needs more points than parameters + 1")
    return n * np.log(max(rss, 1e-300) / n) + 2 * m + 2 * m * (m + 1) / (n - m - 1)


def compare_models(T, k):
    """AICc of the straight line (2 parameters) and of the two channels (4). Needs at least 6 points."""
    T, k = np.asarray(T, dtype=float), np.asarray(k, dtype=float)
    n = T.size
    if n < 6:
        raise ValueError("model comparison needs at least 6 points")
    fl, f2 = line_fit(T, k), fit_two_channel(T, k)
    a1, a2 = aicc(fl["rss"], n, 2), aicc(f2["rss"], n, 4)
    return {"aicc_line": float(a1), "aicc_two": float(a2), "delta": float(a1 - a2), "s_line": fl["s"], "s_two": f2["s"]}


def sign_runs(res):
    """Number of runs of equal sign in the residuals ordered by temperature (a curve gives few runs)."""
    sg = np.sign(res)
    return int(1 + np.sum(sg[1:] != sg[:-1]))

## Step 6: validate against known truth
Repeat the whole experiment many times with new noise and count (a) how often the prediction at
2000 K lands within a factor 1.5 of the truth (the goal), and (b) how often the 95 percent
interval contains the truth (a correct method gives about 0.95).

In [ ]:
def trials(T, T_target=T_TARGET, rel_noise=0.03, repeats=1000, seed=100, p=P_TRUE):
    """Repeat experiment+fit. Returns dict: within15, coverage, median_ratio (fit/true), median_factor (upper/lower)."""
    T = np.asarray(T, dtype=float)
    true = float(two_channel_k(T_target, p))
    w15 = cov_hit = 0
    ratios, facs = [], []
    for i in range(repeats):
        k = synthetic(T, rel_noise, seed + i, p)
        try:
            f = fit_two_channel(T, k)
            m, lo, hi = (v[0] for v in band(f, T_target))
        except (ValueError, np.linalg.LinAlgError, FloatingPointError):
            ratios.append(np.nan)
            facs.append(np.nan)
            continue
        if not (np.isfinite(m) and np.isfinite(lo) and np.isfinite(hi)):
            ratios.append(np.nan)
            facs.append(np.nan)
            continue
        r = m / true
        w15 += int(1.0 / 1.5 < r < 1.5)
        cov_hit += int(lo <= true <= hi)
        ratios.append(r)
        facs.append(hi / lo)
    return {"within15": w15 / repeats, "coverage": cov_hit / repeats,
            "median_ratio": float(np.nanmedian(ratios)), "median_factor": float(np.nanmedian(facs))}

## Step 7: three failures
(a) **The data stop too cool.** If the hottest measurement is below the crossover (1119 K),
the second channel barely shows. The fit then has a wide range of E2 that all describe the
data, and the prediction at 2000 K swings by orders of magnitude. The band is the
warning, and `trials` over different hottest temperatures draws the line.
(b) **A bad start.** A single start can end in a poorer valley; the three start guesses
make that rare, and the RSS tells you which one won.
(c) **Too few points.** Four parameters need at least five points; with five or six the
t factor for 1 or 2 degrees of freedom makes the band very wide, honestly.

In [ ]:
def reach_study(t_max_list, t_min=500.0, n=12, repeats=300, seed=100):
    """For data spanning t_min..t_max, the share within a factor 1.5 at 2000 K and the median band factor."""
    out = []
    for tm in t_max_list:
        r = trials(np.linspace(t_min, tm, n), repeats=repeats, seed=seed)
        out.append((float(tm), r["within15"], r["coverage"], r["median_factor"]))
    return out

## Step 8: see it
Left: the data, the straight line, the two-channel fit and its band out to 2000 K.
Middle: the two channels and their sum; the crossover is where they meet.
Right: the share of repeated experiments that land within a factor 1.5, against the hottest
temperature measured.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    T = T_DATA
    k = synthetic(T, 0.03, seed=0)
    f = fit_two_channel(T, k)
    fl = line_fit(T, k)
    pt = param_table(f)
    mid, lo, hi = band(f, T_TARGET)
    true = float(two_channel_k(T_TARGET))
    print("p", np.round(f["p"], 3), "se", np.round(pt["se"], 3), "s", round(f["s"], 4), "df", f["df"])
    print("k(2000 K)", mid[0], lo[0], hi[0], "true", true, "line", float(line_k(fl, T_TARGET)))
    print("crossover", crossover_T(f["p"]), "cond", condition_number(f, T))
    print("model comparison", compare_models(T, k))
    print("MC", mc_predict(f, T_TARGET))

    fig, ax = plt.subplots(1, 3, figsize=(13, 3.9))
    Tg = 1000.0 / np.linspace(0.45, 2.05, 160)
    m, l, h = band(f, Tg)
    ax[0].fill_between(1000 / Tg, np.log(l), np.log(h), alpha=0.3, label="95 percent band")
    ax[0].plot(1000 / Tg, np.log(m), label="two-channel fit")
    ax[0].plot(1000 / Tg, np.log(line_k(fl, Tg)), "--", label="straight line")
    ax[0].plot(1000 / Tg, np.log(two_channel_k(Tg)), "k:", label="true law")
    ax[0].plot(1000 / T, np.log(k), "o", color="tab:red", label="data")
    ax[0].errorbar([1000.0 / T_TARGET], [np.log(mid[0])], yerr=[[np.log(mid[0] / lo[0])], [np.log(hi[0] / mid[0])]],
                   fmt="s", color="tab:green", capsize=4, label="2000 K, 95 percent")
    ax[0].set_xlabel("1000 / T (1/K x 1000)")
    ax[0].set_ylabel("ln k (k in 1/s)")
    ax[0].set_title("The line misses 2000 K, the fit does not")
    ax[0].legend(fontsize=8, loc="lower left")
    ax[0].grid(alpha=0.3)

    xx = 1000.0 / Tg
    ax[1].plot(xx, (P_TRUE[0] - P_TRUE[1] * xx / R_GAS), label="channel 1 (E = 50)")
    ax[1].plot(xx, (P_TRUE[2] - P_TRUE[3] * xx / R_GAS), label="channel 2 (E = 200)")
    ax[1].plot(xx, np.log(two_channel_k(Tg)), "k", lw=2, label="sum")
    ax[1].axvline(1000.0 / crossover_T(P_TRUE), color="gray", ls=":")
    ax[1].set_ylim(-6, 22)
    ax[1].set_xlabel("1000 / T (1/K x 1000)")
    ax[1].set_ylabel("ln k (k in 1/s)")
    ax[1].set_title("Two lines add up to a curve")
    ax[1].legend(fontsize=8, loc="lower left")
    ax[1].grid(alpha=0.3)

    rs = reach_study([700, 800, 900, 1000, 1100, 1200, 1300, 1450, 1600], repeats=150)
    ax[2].plot([r[0] for r in rs], [100 * r[1] for r in rs], "o-")
    ax[2].axvline(crossover_T(P_TRUE), color="gray", ls=":")
    ax[2].set_xlabel("hottest measured temperature (K)")
    ax[2].set_ylabel("predictions within factor 1.5 (%)")
    ax[2].set_title("Measure past the crossover")
    ax[2].grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "curved_fit.png", dpi=120)
    print("trials 500-1600 K:", trials(T, repeats=600))
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Print `param_table(f)["corr"]` for `T_DATA` and for `np.linspace(500, 1100, 12)` (seed 0, 3 percent error) and explain what happens to the correlation of lnA2 and E2.
2. **Stretch.** Run `reach_study([1200, 1250, 1300, 1350], repeats=300)` and find the lowest hottest temperature at which at least 95 percent of predictions are within a factor 1.5; then check that the median interval factor agrees.
3. **Explain.** Explain, in your own words and with the AICc formula, why a four-parameter fit always has an RSS no larger than a two-parameter fit, and why AICc can still prefer the straight line.

## Assessment
- **A. Conceptual.** Why does adding two Arrhenius terms bend the plot of ln k against 1000/T, and why does the hot end follow the channel with the larger Ea?
- **B. Numerical.** For the true law (lnA1 = 16.118, E1 = 50 kJ/mol, lnA2 = 32.236, E2 = 200 kJ/mol) find the crossover temperature.
- **C. Graph and simulation interpretation.** In the right panel of the figure, the share of correct predictions is about 1 percent when the data stop at 900 K, yet the 95 percent band is narrow there (a factor 1.26). Which number from the fit says that the straight line is enough, and why can that number not rescue the prediction?
- **D. Design and optimisation challenge.** Design: you have 12 points from 500 K at 3 percent error and may choose the hottest temperature. How hot must you go so that at least 95 percent of repeated experiments predict k(2000 K) within a factor 1.5 of the truth? Test the cases in the app.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    T = L.T_DATA
    k = L.synthetic(T, 0.03, 0)
    f = L.fit_two_channel(T, k)
    pt = L.param_table(f)
    true = float(L.two_channel_k(2000.0))
    mid, lo, hi = (v[0] for v in L.band(f, 2000.0))
    # ---- the roadmap checkpoint -----------------------------------------------------------------
    checks.append(("roadmap checkpoint: true k(2000 K) = 5.98e8 per s; 12 points at 3 percent noise, 500-1600 K, predict 5.70e8, within a factor 1.5",
                   abs(true / 5.984e8 - 1) < 1e-3 and abs(mid / 5.704e8 - 1) < 0.002 and 1 / 1.5 < mid / true < 1.5))
    fl = L.line_fit(T, k)
    ratio_line = true / float(L.line_k(fl, 2000.0))
    checks.append(("the straight line on the same data is 51 times too low (Session 3's 800-1600 K data: 9.2 times)", abs(ratio_line - 50.8) < 0.2))
    T3 = np.linspace(800.0, 1600.0, 9)
    k3 = L.synthetic(T3, 0.03, 2)
    checks.append(("Session 3 data (9 points, 800-1600 K): line 9.23 times too low, two-channel fit within 2 percent",
                   abs(true / float(L.line_k(L.line_fit(T3, k3), 2000.0)) - 9.23) < 0.02 and abs(L.band(L.fit_two_channel(T3, k3), 2000.0)[0][0] / true - 0.986) < 0.002))
    # ---- the fit ---------------------------------------------------------------------------------
    checks.append(("worked example: lnA1 = 16.130, E1 = 50.05, lnA2 = 32.099, E2 = 198.51 kJ/mol (truth 16.118, 50, 32.236, 200)",
                   np.allclose(f["p"], [16.1303, 50.0460, 32.0986, 198.5080], atol=0.002) and abs(L.P_TRUE[0] - 16.118) < 1e-3))
    checks.append(("worked example: s = 0.0217 on 8 degrees of freedom (true noise 0.03)", abs(f["s"] - 0.02167) < 2e-5 and f["df"] == 8))
    checks.append(("standard errors: 0.041, 0.228, 0.102 and 1.186", np.allclose(pt["se"], [0.0406, 0.2283, 0.1017, 1.1857], atol=0.0006)))
    checks.append(("noise-free data: the fit returns the true parameters", np.allclose(L.fit_two_channel(T, L.two_channel_k(T))["p"], L.P_TRUE, atol=1e-6)))
    checks.append(("crossover: truth 1119 K, fit 1118 K; the hot channel carries 50 percent there, 99.9 percent at 2000 K, nothing at 500 K",
                   abs(L.crossover_T(L.P_TRUE) - 1119.29) < 0.01 and abs(L.crossover_T(f["p"]) - 1118.21) < 0.01
                   and abs(float(L.channel_share(L.P_TRUE, 1119.29)) - 0.5) < 1e-3 and float(L.channel_share(L.P_TRUE, 2000.0)) > 0.999 and float(L.channel_share(L.P_TRUE, 500.0)) < 1e-6))
    sw = L.fit_two_channel(T, k, starts=[[32.0, 200.0, 16.0, 50.0]])
    checks.append(("channels are labelled by Ea: a start with the channels swapped still returns E1 < E2 and the same fit", sw["p"][1] < sw["p"][3] and np.allclose(sw["p"], f["p"], atol=1e-6)))
    bad = L.fit_two_channel(T, k, starts=[[20.0, 300.0, 20.0, 20.0]])
    checks.append(("a single bad start gets stuck (RSS 16.1); the three built-in starts reach RSS 0.00376", bad["rss"] > 10.0 and abs(f["rss"] - 0.003756) < 2e-6 and f["n_starts"] == 3))
    # ---- covariance ------------------------------------------------------------------------------
    checks.append(("identifiability: lnA and E of the same channel are correlated, 0.974 and 0.995", abs(pt["corr"][0, 1] - 0.974) < 0.002 and abs(pt["corr"][2, 3] - 0.9946) < 0.001))
    checks.append(("condition number of the scaled J^T J is 460 here, and far higher when the data stop at 1000 K",
                   abs(L.condition_number(f, T) - 459.5) < 1.0 and L.condition_number(L.fit_two_channel(np.linspace(500, 1000, 12), L.synthetic(np.linspace(500, 1000, 12), 0.03, 0)), np.linspace(500, 1000, 12)) > 5000))
    # ---- band ------------------------------------------------------------------------------------
    se = float(L.se_ln_k(f, [2000.0])[0])
    checks.append(("worked example: se of ln k at 2000 K = 0.0317 and the 95 percent interval is 5.30e8 to 6.14e8 (factor 1.157), true value inside",
                   abs(se - 0.03167) < 2e-5 and abs(lo / 5.3027e8 - 1) < 0.001 and abs(hi / 6.1366e8 - 1) < 0.001 and abs(hi / lo - 1.1573) < 0.001 and lo < true < hi))
    med, mlo, mhi = L.mc_predict(f, 2000.0)
    checks.append(("Monte Carlo from the covariance agrees with the delta-method band within 1.5 percent",
                   abs(med / mid - 1) < 0.005 and abs(mlo / lo - 1) < 0.015 and abs(mhi / hi - 1) < 0.015))
    checks.append(("the band is narrowest inside the data and flares outside: factor at 1000 K, 1600 K and 2000 K increases",
                   (lambda r: r[0] < r[1] < r[2])([float(L.band(f, t)[2][0] / L.band(f, t)[1][0]) for t in (1000.0, 1600.0, 2000.0)])))
    # ---- model comparison ------------------------------------------------------------------------
    cm = L.compare_models(T, k)
    checks.append(("model comparison: AICc 8.8 for the line and -83.1 for two channels (difference 92 in favour of two channels)",
                   abs(cm["aicc_line"] - 8.828) < 0.01 and abs(cm["aicc_two"] + 83.116) < 0.01 and abs(cm["delta"] - 91.94) < 0.02))
    rl = L.sign_runs(fl["res"])
    r2 = L.sign_runs(np.log(k) - L.ln_k(f["p"], T))
    checks.append(("the line's residuals come in 3 runs of one sign (a curve), the two-channel residuals in 7 (noise); s is 1.267 against 0.0217",
                   rl == 3 and r2 == 7 and abs(fl["s"] - 1.2672) < 0.001))
    # ---- validation ------------------------------------------------------------------------------
    r = L.trials(T, repeats=1000)
    checks.append((f"validation: 1000 repeated experiments, {100 * r['within15']:.0f} percent within a factor 1.5, interval holds the truth {100 * r['coverage']:.1f} percent (about 95), median factor {r['median_factor']:.2f}",
                   r["within15"] > 0.99 and abs(r["coverage"] - 0.945) < 0.001 and 0.93 < r["coverage"] < 0.97 and abs(r["median_factor"] - 1.215) < 0.005))
    r6 = L.trials(np.linspace(500.0, 1600.0, 6), repeats=150)
    r5 = L.trials(np.linspace(500.0, 1600.0, 5), repeats=150)
    checks.append((f"few points: 6 points give median factor {r6['median_factor']:.2f}, 5 points {r5['median_factor']:.1f} (1 degree of freedom), still honest",
                   abs(r6["median_factor"] - 1.51) < 0.02 and abs(r5["median_factor"] - 3.35) < 0.05 and r5["coverage"] > 0.9 and r5["within15"] > 0.95))
    r20 = L.trials(T, rel_noise=0.20, repeats=300)
    checks.append((f"noise: at 20 percent error only {100 * r20['within15']:.0f} percent of predictions are within 1.5, the band widens to {r20['median_factor']:.1f} and the coverage stays {100 * r20['coverage']:.0f} percent",
                   abs(r20["within15"] - 0.82) < 0.01 and abs(r20["median_factor"] - 3.69) < 0.05 and r20["coverage"] > 0.92))
    # ---- failures --------------------------------------------------------------------------------
    rs = {int(t): v for t, *v in L.reach_study([900, 1100, 1300], repeats=150)}
    checks.append((f"failure: data stop at 900 K: {100 * rs[900][0]:.1f} percent within 1.5, and the interval holds the truth only {100 * rs[900][1]:.0f} percent: the model cannot see a channel the data never reach",
                   rs[900][0] < 0.03 and abs(rs[900][1] - 0.22) < 0.01))
    checks.append((f"data stop at 1100 K (about the crossover): {100 * rs[1100][0]:.0f} percent within 1.5, honest but huge band (median factor {rs[1100][2]:.0f}); at 1300 K: {100 * rs[1300][0]:.0f} percent",
                   abs(rs[1100][0] - 0.287) < 0.01 and rs[1100][2] > 50 and rs[1100][1] > 0.92 and rs[1300][0] > 0.98))
    d = []
    for tm in (900, 1100):
        Tm = np.linspace(500.0, tm, 12)
        d.append(np.median([L.compare_models(Tm, L.synthetic(Tm, 0.03, 100 + i))["delta"] for i in range(40)]))
    checks.append((f"AICc as a warning: when the data stop at 900 K the line wins (median difference {d[0]:.1f}); at 1100 K the two channels win ({d[1]:.0f})",
                   d[0] < 0 and d[1] > 10))
    # ---- mini-project ----------------------------------------------------------------------------
    pm = np.array([np.log(3e6), 60.0, np.log(2e12), 170.0])
    Tm = np.linspace(600.0, 1500.0, 10)
    km = L.synthetic(Tm, 0.04, 9, pm)
    fm = L.fit_two_channel(Tm, km)
    qm = L.param_table(fm)
    mm, lm, hm = (v[0] for v in L.band(fm, 2000.0))
    tm_true = float(L.two_channel_k(2000.0, pm))
    checks.append(("mini-project data: Ea1 = 60.04 +- 0.83 and Ea2 = 167.8 +- 1.57 kJ/mol, crossover 980 K (truth 987 K)",
                   abs(fm["p"][1] - 60.04) < 0.02 and abs(qm["se"][1] - 0.83) < 0.02 and abs(fm["p"][3] - 167.80) < 0.03 and abs(qm["se"][3] - 1.57) < 0.02
                   and abs(L.crossover_T(fm["p"]) - 980.3) < 0.2 and abs(L.crossover_T(pm) - 986.6) < 0.2))
    checks.append(("mini-project: k(2000 K) = 6.75e7 per s with interval 5.95e7 to 7.67e7, true 7.27e7 inside; the line would be 13 times too low",
                   abs(mm / 6.752e7 - 1) < 0.002 and abs(lm / 5.947e7 - 1) < 0.002 and abs(hm / 7.666e7 - 1) < 0.002 and lm < tm_true < hm
                   and abs(tm_true / 7.272e7 - 1) < 0.001 and abs(tm_true / float(L.line_k(L.line_fit(Tm, km), 2000.0)) - 13.43) < 0.1))
    ws = L.trials(np.linspace(600.0, 1000.0, 10), rel_noise=0.04, repeats=150, p=pm)
    checks.append((f"mini-project lever: stop at 1000 K and only {100 * ws['within15']:.0f} percent of predictions are within 1.5 (full window: 100 percent)",
                   ws["within15"] < 0.4 and L.trials(Tm, rel_noise=0.04, repeats=150, p=pm)["within15"] > 0.99))
    # ---- edge cases ------------------------------------------------------------------------------
    raised = []
    for fn in (lambda: L.fit_two_channel(T[:4], k[:4]), lambda: L.compare_models(T[:5], k[:5])):
        try:
            fn()
            raised.append(False)
        except ValueError:
            raised.append(True)
    checks.append(("edge cases: 4 points cannot fit 4 parameters with an error estimate, and 5 points cannot be compared by AICc: both refused", all(raised)))
    five = L.fit_two_channel(T[::3].tolist() + [1550.0], L.synthetic(T[::3].tolist() + [1550.0], 0.03, 3))
    checks.append(("edge case: 5 points give 1 degree of freedom and t = 12.706", five["df"] == 1 and abs(L.t95(five["df"]) - 12.706) < 1e-9))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
